In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ellc
import edmcmc as edm
import corner
import pandas as pd
from matplotlib import gridspec
from matplotlib.pyplot import cm
import batman
from astropy.io import ascii
import math
from scipy.optimize import least_squares
from scipy.stats import linregress

settings = np.seterr(over="ignore")

params = {
    "axes.labelsize": 18,
    "axes.labelpad": 9,
    "axes.titlesize": 20,
    "axes.linewidth": 2,
    "axes.labelweight": 3,
    "axes.titleweight": 3,
    "font.size": 15,
    "legend.fontsize": 15,
    "lines.linewidth": 2,
    "xtick.major.width": 2,
    "xtick.minor.width": 1,
    "xtick.major.size": 8,
    "xtick.minor.size": 5,
    "xtick.major.pad": 5,
    "xtick.labelsize": 15,
    "xtick.minor.visible": True,
    "xtick.direction": "in",
    "xtick.top": True,
    "ytick.major.width": 2,
    "ytick.minor.width": 1,
    "ytick.major.size": 8,
    "ytick.minor.size": 5,
    "ytick.major.pad": 7,
    "ytick.labelsize": 15,
    "ytick.minor.visible": True,
    "ytick.direction": "in",
    "ytick.right": True,
    "legend.frameon": True,
    "legend.loc": "upper right",
    #'text.usetex': True,
    #'text.latex.preamble': '\usepackage{helvet}\usepackage[T1]{fontenc}\usepackage{sfmath}',
    #'font.sans-serif': "Helvetica",
    #'font.family': "sans-serif",
    "ps.usedistiller": "xpdf",
    "savefig.dpi": 300,
    "figure.figsize": [7, 7],
}

plt.rcParams.update(params)

In [ ]:
# adding noise to transit
def std_dev(arr_time):
    t_or_f = arr_time < -0.25
    arr_mask_1 = arr_time[t_or_f]
    masking2 = arr_time > 0.175
    arr_mask_2 = arr_time[masking2]

    # finding the values where the noise starts
    val1 = arr_mask_1[-1]
    val2 = arr_mask_2[0]

    # index where it occurs
    idx1 = np.where(arr_time == val1)
    idx2 = np.where(arr_time == val2)
    return idx1, idx2

In [ ]:
#Transit (cleaned)
transit_file = ascii.read("/home/nadja/Documents/UWMadison/Research/TOI5082/tic437011608flattened-2min.csv")

time = transit_file["Time (BJD-2457000)"] 
transit_flux = transit_file["Flattened Flux"]

transit_time = time + 2457000

In [ ]:
def unpack_file(filepath):
    df = pd.read_csv(filepath, comment="#")
    for col in ("ccfjdsum", "ccfrvmod", "dvrms"):
        if col not in df.columns:
            raise ValueError(f"Input CSV missing required column: {col}")

    time_rv = df["ccfjdsum"].values.astype(float)  # times in days
    data_rv = df["ccfrvmod"].values.astype(float)  # observed RV in km/s
    err_rv = df["dvrms"].values.astype(float)  # RV uncertainties in km/s
    
    data_rv -= np.median(data_rv)
    return time_rv, data_rv, err_rv

In [ ]:
def sma_star_units(sma, rstar):
    return (sma*rstar) #rstar in units of Rsun

In [ ]:
## Input files and output dir

directory = '/home/nadja/Documents/UWMadison/Research/TOI5082/'
second_rv = (directory + "2026-01-20_TOI5082.csv")
time_obs_2, rv_data_2, rv_err_2 = unpack_file(second_rv)
write_chains = True
thin_n = 5


#mcmc
nlink = 25000
nburnin = 4000 

In [ ]:
#initial guesses
t0_bjd = 2459508.8190741274
rp_rs = (10 ** (-3) * 0.93**2) ** 0.5  # stellar radius units
sma = 11.8  #RATIO
RSTAR = 0.93
a_calc = sma_star_units(sma, RSTAR)
vsini = 5
lambda_guess = -20
incl = 87
r_1 = 1 / sma
r_2 = rp_rs * r_1
orbital_period = 4.2403567  # orbital_period
q_guess = 0.00002109

shape = 'sphere'
e_val = 0
periastron = 90
coef_1 = 0.1
coef_2 = 0.3

fs_guess = np.sqrt(e_val)*math.sin(np.deg2rad(periastron))
fc_guess  = np.sqrt(e_val)*math.cos(np.deg2rad(periastron))

p_star_guess = 1.743589

#num_events_1 = int(np.round((rv_t0-trans_t0)/orbital_period))
#num_events_2 = int(np.round((rv_t0_2 - trans_t0)/orbital_period))
#K_guess = 0.5*(np.nanmax(rv_data_1)-np.nanmin(rv_data_1))

labels=["a_over_R", "radius_planet", "vsini", "obliquity", "t0_trans", "incl", "period", 'fs', 'fc', 'c1', 'c2', 'slope', 'intercept']

In [ ]:
#systemic offset
i_rad = np.radians(incl)
rsum = (r_1 + r_2)
val = rsum / max(1e-12, np.sin(i_rad))
if val >= 1.0:
    transit_duration_days = 0.2
else:
    transit_duration_days = orbital_period / np.pi * val

transit_half_phase = (transit_duration_days / 2.0) / orbital_period
phases_for_mask = ((time_obs_2 - t0_bjd) / orbital_period + 0.5) % 1.0 - 0.5 #change
in_transit_mask = np.abs(phases_for_mask) < transit_half_phase
out_of_transit_mask = ~in_transit_mask

if out_of_transit_mask.sum() < 3:
    out_of_transit_mask = np.ones_like(out_of_transit_mask, dtype=bool)
weights = 1.0 / (rv_err_2**2) #change

gamma_weighted = np.sum(weights[out_of_transit_mask] * rv_data_2[out_of_transit_mask]) / np.sum(weights[out_of_transit_mask]) #change
rv_data_2 = rv_data_2 - gamma_weighted #change

In [ ]:
#TESS standard dev
phase_trans = ((transit_time - t0_bjd) / orbital_period) - np.round(
    (transit_time - t0_bjd) / orbital_period
)

i, j = std_dev(phase_trans)

idx1 = i[0][0]
idx2 = j[0][0]

noise_arr = np.concatenate((transit_flux[:idx1], transit_flux[idx2:]), axis=0)
error = np.nanstd(noise_arr)
trans_err = np.full_like(transit_flux, error)

In [ ]:
#linear background trend
time_centered_1 = time_obs_2 - np.median(time_obs_2)
slope, intercept_new, r_val, p_val, std_err = linregress(time_centered_1, rv_data_2)

In [ ]:
def dumb_loglikelihood(p, time, data):
    vsini_guess, obliq = p
    rv_model, _ = ellc.rv(
        t_obs=time,
        t_zero=t0_bjd, #change value
        period=orbital_period,
        lambda_1=obliq,
        radius_1=r_1,
        radius_2=r_2,
        incl=incl,
        a=sma_star_units(sma, RSTAR),
        f_c=fc_guess,
        f_s=fs_guess,
        shape_1=shape,
        shape_2=shape,
        vsini_1=vsini_guess,
        flux_weighted=True,
        sbratio=0,
        q=q_guess,)
    base = rv_model + (slope * time_centered_1 + intercept_new) 
    res = data - base
    return res

#using lm for initial guesses of system
p1 = [vsini, lambda_guess]
#bnds = ((0, 10), (-180, 180))
soln = least_squares(
    dumb_loglikelihood, p1, args=(time_obs_2, rv_data_2), method="lm")
x = soln.x

vsini_calc = x[0]
obl_calc = x[1]

In [ ]:
# Commented here is the edmcmc I made for batman. Dunno if this helps
def lnprior(p):
    # a will be the data we place while mu will be the literature values
    # have a be the closest to real data a/R*, Rp/R*
    r_star_sma, rplanet, vsini_guess, obliquity, t0_trans, inc, period, fs, fc, c_1, c_2, a, c = p

    r_1 = 1/r_star_sma
    r_2 = r_1 * rplanet
    if t0_trans < t0_bjd - 1 or t0_trans >  t0_bjd+1:
        return -np.inf
    '''if period > orbital_period + 0.0001 or period < orbital_period - 0.0001:
        return -np.inf
    if (r_1+r_2) > 1 or (r_1+r_2) < 0:
        return -np.inf'''
    if rplanet <= 0 or rplanet >= 1:
        return -np.inf
    if inc < 0 or inc > 90:
        return -np.inf
    if fs <=-1 or fs >= 1:
        return -np.inf
    if fc <=-1 or fc >= 1:
        return -np.inf
    if c_1 > 1 or c_1 <0:
        return -np.inf
    if c_2 > 1 or c_2 < 0:
        return -np.inf
    if obliquity > 180 or obliquity < -180:
        return -np.inf
    if vsini_guess < 0 or vsini_guess > 20:
        return -np.inf
    w_peri = np.arctan2(fs, fc)
    ecc = fs**2 +fc**2
    if ecc >=0.975:
        return -np.inf
    if not np.isfinite(ecc):
        return -np.inf
    b = np.abs(r_star_sma * np.cos(np.deg2rad(inc)) * (1-ecc**2)/(1+ecc*np.sin(w_peri)))
    if b < 0 or b > 1:
        return -np.inf
    if not (-0.5 < a < 0.5): return -np.inf
    #if not (-100000.0 < c < 100000.0): return -np.inf

    p_star = (3* np.pi * pow(r_star_sma,3))/(497.582 * period**2)
    """
    give ln prior all of the parameters, then separate each to spit back to 
    loglikelihood function. lnprior gets a/R*, Rp/R* and their respective errs
    """

    mu_p_star = 1.743589  # 24.2
    sigma_p = 0.1

    # Calculate individual priors
    prior_pstar = (
        np.log(1.0 / (np.sqrt(2 * np.pi) * sigma_p))
        - 0.5 * (p_star - mu_p_star) ** 2 / sigma_p**2
    )

    return prior_pstar

def trans_loglikelihood(p, trans_time, trans_obs, err):
    r_star_sma, rplanet, vsini_guess, obliquity, t0_trans, inc, period, fs, fc, c_1, c_2, a, c = p
    ecc = fs **2 + fc **2 
    
    w_peri = np.arctan2(fs, fc)
    
    params = batman.TransitParams()
    params.t0 = t0_trans # time of inferior conjunction
    params.rp = rplanet  # planet radius (in units of stellar radii)
    params.a = r_star_sma  # semi-major axis (in units of stellar radii)
    params.inc = inc  # orbital inclination (in degrees)
    params.per = period  # orbital period
    params.ecc = ecc  # eccentricity

    params.w = np.rad2deg(w_peri)  # longitude of periastron (in degrees)
    params.u = [c_1, c_2]  # limb darkening coefficients [u1, u2]
    params.limb_dark = "quadratic"  # limb darkening model

    try: 
        m = batman.TransitModel(params, trans_time)
        model = m.light_curve(params)
    except Exception:
        return -np.inf
    
    norm = np.sum(np.log(2*np.pi*err**2))
    chisq = np.sum((trans_obs - model) ** 2 / err**2)

    trans_loglikelihood = -0.5 * (chisq+norm)
    return trans_loglikelihood

def rv_loglikelihood(p, time, rv_obs, rv_err):
    # may need to add a linear model
    r_star_sma, rplanet, vsini_guess, obliquity, t0_trans, inc, period, fs, fc, c_1, c_2, a, c = p
    r1 = 1 / r_star_sma
    r2 = rplanet * r1

    semi_major_axis = sma_star_units(r_star_sma, RSTAR)
    rv_model, _ = ellc.rv(
        time,
        t_zero=t0_trans,
        period=period,
        lambda_1=obliquity,
        radius_1=r1,
        radius_2=r2,
        incl=inc,
        f_s=fs,
        f_c=fc,
        a=semi_major_axis,
        shape_1="sphere",
        shape_2="sphere",
        vsini_1=vsini_guess,
        flux_weighted=True,
        sbratio=0,
        q=q_guess,
        verbose=0
        )
    
    time_centered = time - np.median(time)
    model_slope = a * time_centered + c

    base = rv_model + model_slope
    norm = np.sum(np.log(2*np.pi*rv_err**2))
    chisq = np.sum((rv_obs - base) ** 2 / rv_err**2)

    rv_loglikelihood = -0.5 * (chisq+norm)
    return rv_loglikelihood

def lnprob(p, time_rv, time_trans, rv_obs, trans_obs, rv_err, trans_err):
    r_star_sma, rplanet, vsini_guess, obliquity, t0_trans, inc, period, fs, fc, c_1, c_2 , a, c = p
    penal_pstar = lnprior(p)
    if not np.isfinite(penal_pstar):
        return -np.inf
    rv_logp = rv_loglikelihood(p, time_rv, rv_obs, rv_err) 
    if not np.all(np.isfinite(rv_logp)):
        return -np.inf
    trans_logp = trans_loglikelihood(p, time_trans, trans_obs, trans_err)
    if not np.all(np.isfinite(trans_logp)):
        return -np.inf
    return (rv_logp + trans_logp + penal_pstar)

In [ ]:
#r_star_sma, rplanet, vsini_guess, obliquity, t0_trans, inc, period, fs, fc = p
p0 = [sma, rp_rs, vsini_calc, obl_calc, t0_bjd, incl, orbital_period, fs_guess, fc_guess, coef_1, coef_2, slope, intercept_new]
#p0 = [sma, rp_rs, vsini_calc, obl_calc, trans_t0, incl, orbital_period, coef_1, coef_2, p_star_guess]
ndim = len(p0)

out = edm.edmcmc(
    lnprob,
    p0,
    [0.01, 0.001, 0.1, 1, 0.00001, 0.001, 0.00001, 0.05, 0.05, 0.001, 0.001, 0.000001, 0.001],
    args=(time_obs_2, transit_time, rv_data_2, transit_flux, rv_err_2, trans_err),
    nwalkers=200,
    nlink=nlink,
    nburnin=nburnin,
    m1mac=False,
    ncores=10,
)

In [ ]:
samples_for_outputs = out.get_chains(nthin=thin_n, nburnin=nburnin, flat=True)

if write_chains:
    np.savez(
        directory + 'first_event_chains_25000_vsini.npz',
        thinflatchains=samples_for_outputs,
        lastpos=out.lastpos,
        nwalkers=out.nwalkers,
        npar=out.npar,
        nburnin=out.nburnin,
        thin_n=thin_n,
        nlink=out.nlink,
        labels=np.array(labels, dtype=object),
)

In [ ]:
thinned_all_samples = out.get_chains(nthin=thin_n, nburnin=nburnin, flat=False)
np.savez(
        directory + 'second_event_allchains.npz',
        allchains=thinned_all_samples,
)

In [ ]:
#prints

for i in range(len(labels)):
    print(f'{labels[i]}: {np.median(out.flatchains[:, i])}+/-{np.nanstd(out.flatchains[:,i])} \n')


In [ ]:
fig = corner.corner(
    out.flatchains,
    labels=["a_over_R", "radius_planet", "vsini", "obliquity", "t0_trans", "incl", "period", 'fs', 'fc', 'c1', 'c2', 'slope', 'intercept'],
)

plt.savefig(directory + '/first_event/corner_first_event.png')
plt.show()
plt.close(fig)

In [ ]:
best_t0_trans = np.median(out.flatchains[:, 4]) 
best_period = np.median(out.flatchains[:, 6])
best_rp = np.median(out.flatchains[:,1])
best_sma = np.median(out.flatchains[:,0])
best_vsini = np.median(out.flatchains[:,2])
best_lambda = np.median(out.flatchains[:, 3])
best_incl = np.median(out.flatchains[:, 5])
best_fs = np.median(out.flatchains[:,7])
best_fc = np.median(out.flatchains[:,8])
best_c1 = np.median(out.flatchains[:,9])
best_c2 = np.median(out.flatchains[:,10])
best_ecc = best_fs**2 + best_fc**2
best_omega = np.rad2deg(np.arctan2(best_fs, best_fc))

# [sma, rp_rs, vsini, lambda_guess, trans_t0, incl, rv_t0, period]
params1 = batman.TransitParams()
params1.t0 = best_t0_trans  # time of inferior conjunction
params1.per = best_period  # orbital period
params1.rp = np.median(out.flatchains[:, 1])  # planet radius (in units of stellar radii)
params1.a = best_sma # semi-major axis (in units of stellar radii)
params1.inc = best_incl# orbital inclination (in degrees)
params1.ecc = best_ecc # eccentricity
params1.w = best_omega # longitude of periastron (in degrees)
params1.u = [best_c1, best_c2]  # limb darkening coefficients [u1, u2]
params1.limb_dark = "quadratic"  # limb darkening model


model = batman.TransitModel(params1, transit_time)
unsorted_flux = model.light_curve(params1)
bat_flux_phase = np.copy(phase_trans)
sort_index = np.argsort(bat_flux_phase)

bat_flux_phase = bat_flux_phase[sort_index]
bat_flux_sorted = unsorted_flux[sort_index]

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8, 8))
ax2.set_ylabel("Flux")
ax2.set_xlabel("Phase")
ax2.set_xlim(-0.05, 0.05)

plt.scatter(phase_trans, transit_flux, color='black')
plt.errorbar(phase_trans, transit_flux, yerr=trans_err, xerr=None, alpha=0.3, fmt="None", c='black')
plt.plot(bat_flux_phase, bat_flux_sorted, color="red")
plt.savefig(directory + '/first_event/toi5082_transit_first_event.png')

In [ ]:
def rv_function(aR_guess, rp_guess, vsini_guess, obliq_guess, incl_guess, t0_guess, period, fs, fc, c1,c2, timeArr):
    t0_rv_guess = t0_guess #change
    rv_model = ellc.rv(
        timeArr,
        t_zero=t0_rv_guess,
        period=period,
        lambda_1=obliq_guess,
        radius_1 = 1/aR_guess,
        radius_2= rp_guess * (1/aR_guess),
        incl=incl_guess,
        a=aR_guess,
        f_c=fc,
        f_s=fs,
        shape_1="sphere",
        shape_2="sphere",
        sbratio=0,
        vsini_1=vsini_guess,
        flux_weighted=True,
        q=0.00002109,
        ldc_1 = [c1,c2],
        ld_1='quad'
    )
    rv_model = np.asarray(rv_model[0])
    return rv_model

def phase_fold(arrTime, event):
    return ((arrTime - event) / orbital_period) - np.round((arrTime - event) / orbital_period)

def sorting(phaseArr, fluxArr):
    sorted_phase = np.copy(phaseArr)
    sort_draw = np.argsort(sorted_phase)  # this returns the indices that would sort the array
    sorted_phase = sorted_phase[sort_draw]
    sorted_flux= fluxArr[sort_draw]
    return sorted_phase, sorted_flux

In [ ]:
phase_rv = ((time_obs_2 - best_t0_trans) / best_period) - np.round(
    (time_obs_2 - best_t0_trans) / best_period)
#change
def unpack_params(arr):
    a_R = arr[0]
    rp = arr[1]
    vsini = arr[2]
    obliq = arr[3]
    t0 = arr[4]
    incl = arr[5]
    per = arr[6]
    f_s = arr[7]
    f_c = arr[8]
    c1 = arr[9]    
    c2 = arr[10]   
    slope = arr[11]
    intercept = arr[12]
    return a_R, rp, vsini, obliq, incl, t0, per, f_s, f_c, c1, c2, slope, intercept

# Posterior models compilation step
samples_for_outputs = out.get_chains(nthin=5, nburnin=nburnin, flat=True)

# aR_guess, rp_guess, vsini_guess, obliq_guess, incl_guess, t0_guess, period, fs, fc, c1,c2, timeArr

# Posterior bands
all_samples = samples_for_outputs
nsamples_total = all_samples.shape[0]
nsamp = min(1000, nsamples_total)
sel_idx = np.linspace(0, nsamples_total - 1, nsamp, dtype=int)

ntime = len(time_obs_2) #change
models = np.zeros((nsamp, ntime))

for j, idx in enumerate(sel_idx):
    unpacked = unpack_params(all_samples[idx, :])
    if unpacked is None:
        models[j, :] = np.nan
        continue
    
    (a_over_s, rp_s, vsini_s, lambda_s, inc_s, t0_bjd_s, 
     per_s, fs_s, fc_s, ld1, ld2, slope_s, intercept_s) = unpacked
    
    #RV from ellc (no bg)
    pure_rv = rv_function(a_over_s, rp_s, vsini_s, lambda_s, inc_s, t0_bjd_s, per_s, fs_s, fc_s, ld1, ld2, time_obs_2) #change
    
    #injecting the bg trend.
    models[j, :] = pure_rv + (slope_s * time_centered_1 + intercept_s)

# Track statistical distribution slices across models
median_model = np.nanmedian(models, axis=0)
p16 = np.nanpercentile(models, 16.0, axis=0)
p84 = np.nanpercentile(models, 84.0, axis=0)
p025 = np.nanpercentile(models, 2.5, axis=0)
p975 = np.nanpercentile(models, 97.5, axis=0)

#plotting
fig3 = plt.figure(figsize=(10, 6))
gs = gridspec.GridSpec(nrows=2, ncols=1, figure=fig3, hspace=0.0, height_ratios=[3, 1])

ax3 = plt.subplot(gs[0, 0])
# Plot data points in phase coordinate space
ax3.errorbar(phase_rv, rv_data_2 * 1e3, yerr=rv_err_2 * 1e3, fmt="o", ms=5, c="k", label="Data", zorder=5)

sort_index = np.argsort(phase_rv)
phase_sorted = phase_rv[sort_index]
model_sorted = median_model[sort_index]
p025_sorted = p025[sort_index]
p975_sorted = p975[sort_index]
p16_sorted = p16[sort_index]
p84_sorted = p84[sort_index]

# Plot corrected line
ax3.plot(phase_sorted, model_sorted * 1e3, "-", lw=1.5, c="blue", label='Median Model', zorder=4)

#1-sig, 2-sig contours
ax3.fill_between(phase_sorted, p025_sorted * 1e3, p975_sorted * 1e3, color='red', alpha=0.2, linewidth=0.0)
ax3.fill_between(phase_sorted, p16_sorted * 1e3, p84_sorted * 1e3, color='red', alpha=0.4, linewidth=0.0)

ax3.set_ylabel("Radial velocity (m/s)", color="black")
ax3.set_ylim(-7.499, 8)
ax3.legend(loc='upper left')

#Residuals
residuals_ms = (rv_data_2 - median_model) * 1e3
ax_bot = plt.subplot(gs[1, 0], sharex=ax3)
ax_bot.errorbar(phase_rv, residuals_ms, yerr=rv_err_2 * 1e3, fmt="o", c="k", ms=4)
ax_bot.axhline(0.0, color="red", alpha=0.7)
ax_bot.set_xlabel("Orbital Phase", color="black")
ax_bot.set_ylabel("Resid", color="black")

plt.savefig(directory + '/first_event/toi5082_rv_first_event.png', bbox_inches='tight', dpi=300)



In [ ]:
gelmanrubinmetrics = out.gelmanrubin()
for i in range(len(gelmanrubinmetrics)):
    print('Parameter number ' + str(i+1) + ' (' + labels[i]+') has a Gelman-Rubin statistic of '
          + str(gelmanrubinmetrics[i])+'.')